# SmartDs Spherical Shell Mass Flux (Starter)

This notebook loads the 3D BATSRUS sample file, samples a spherical shell at `R=10`, computes the radial mass flux on that shell, and plots it as a 2D map.

It reuses the **library** for shell sampling and mass-loss integration (no VTK/PyVista, no custom resampling).

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm

from starwinds_analysis.smart_ds import SmartDs
from starwinds_analysis.analysis.shells import (
    infer_body_radius_m,
    integrate_shell_scalar,
    resolve_batsrus_density_si,
    resolve_batsrus_vector_xyz_si,
    sample_spherical_shells,
)
from starwinds_analysis.analysis.mass_loss import mass_loss_vs_radius
from starwinds_analysis.recipes.spherical import spherical_vector_components

plt.rcParams['figure.dpi'] = 120


## Load the 3D Sample File

Defaults to `sample_data/3d__var_3_n00060000.plt`.

In [ ]:
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'sample_data').exists():
    REPO_ROOT = Path('/Users/dagfev/Documents/starwinds/starwinds-analysis')

DATA_FILE = REPO_ROOT / 'sample_data' / '3d__var_3_n00060000.plt'
if not DATA_FILE.exists():
    candidates = sorted((REPO_ROOT / 'sample_data').glob('3d*00060000*.plt'))
    if not candidates:
        available = sorted(p.name for p in (REPO_ROOT / 'sample_data').glob('*.plt'))
        raise FileNotFoundError('3D n00060000 sample not found. Available .plt files: ' + ', '.join(available))
    DATA_FILE = candidates[0]

print('Using:', DATA_FILE)
sds = SmartDs.from_file(str(DATA_FILE))
print('Title:', sds.title)
print('Zone :', sds.zone)
print('Points:', len(sds.points))
print('Variables:', len(sds.variables))


## Sample a Spherical Shell at `R = 10` and Compute Mass Flux

The shell sampling uses the library grid sampler (for plotting on a regular `theta/phi` grid).

In [ ]:
R_SHELL = 10.0  # in BATSRUS coordinate units (typically body radii)
N_POLAR = 48
N_AZIMUTH = 96

body_radius_m = infer_body_radius_m(sds)
print(f"Body radius [m]: {body_radius_m:.6e}")

rho_name, rho_scale = resolve_batsrus_density_si(sds)
(ux_name, uy_name, uz_name), u_scale = resolve_batsrus_vector_xyz_si(sds, "U")
print("Density field:", rho_name, "-> SI scale", rho_scale)
print("Velocity fields:", (ux_name, uy_name, uz_name), "-> SI scale", u_scale)

shell = sample_spherical_shells(
    sds,
    [R_SHELL],
    fields=(rho_name, ux_name, uy_name, uz_name),
    n_polar=N_POLAR,
    n_azimuth=N_AZIMUTH,
    method="nearest",
    length_unit_to_m=body_radius_m,
)

rho = rho_scale * shell.fields[rho_name]
ux = u_scale * shell.fields[ux_name]
uy = u_scale * shell.fields[uy_name]
uz = u_scale * shell.fields[uz_name]

u_r, _u_theta, _u_phi = spherical_vector_components(ux, uy, uz, shell.x, shell.y, shell.z)
mass_flux = rho * u_r  # kg / m^2 / s

mass_flux_shell = mass_flux[0]
mass_flux_finite = mass_flux_shell[np.isfinite(mass_flux_shell)]
print("Mass-flux finite cells:", mass_flux_finite.size, "/", mass_flux_shell.size)
print("Mass-flux range [kg/m^2/s]:", float(np.nanmin(mass_flux_finite)), float(np.nanmax(mass_flux_finite)))


In [ ]:
# 2D shell plot in longitude/latitude using the native theta/phi sampling grid
lon_deg = np.degrees(shell.phi)
lat_deg = 90.0 - np.degrees(shell.theta)

finite = np.isfinite(mass_flux_shell)
abs_max = float(np.nanpercentile(np.abs(mass_flux_shell[finite]), 99.0)) if finite.any() else 1.0
abs_max = max(abs_max, 1e-30)
linthresh = max(abs_max * 1e-3, 1e-30)
norm = SymLogNorm(linthresh=linthresh, vmin=-abs_max, vmax=abs_max)

fig, ax = plt.subplots(figsize=(9, 4.5))
img = ax.pcolormesh(lon_deg, lat_deg, mass_flux_shell, shading='nearest', cmap='coolwarm', norm=norm)
ax.set_xlabel("Longitude [deg]")
ax.set_ylabel("Latitude [deg]")
ax.set_title(f"Shell mass flux at R={R_SHELL:g} [kg/m^2/s]")
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
fig.colorbar(img, ax=ax, label="Mass flux [kg/m^2/s]")
plt.show()


## Mass-Loss Rate at `R = 10` (Library Function)

This uses `mass_loss_vs_radius(...)`, which defaults to Fibonacci-sphere sampling for the shell integral.

In [ ]:
# Optional cross-check: integrate the plotted grid-shell mass flux directly
mass_loss_grid, coverage_grid = integrate_shell_scalar(mass_flux, shell.area)

profile = mass_loss_vs_radius(
    sds,
    [R_SHELL],
    n_polar=N_POLAR,
    n_azimuth=N_AZIMUTH,
    sampling="fibonacci",
    method="nearest",
)

dotm_lib = float(profile["mass_loss [kg/s]"][0])
cov_lib = float(profile["coverage [none]"][0])

print(f"Grid-shell integral (same plotted shell): {float(mass_loss_grid[0]):.6e} kg/s")
print(f"Grid-shell coverage             : {float(coverage_grid[0]):.6f}")
print(f"Library mass_loss_vs_radius     : {dotm_lib:.6e} kg/s")
print(f"Library coverage                : {cov_lib:.6f}")
